# Phase 3: Credit Risk Model Development
## Step 2: Tree-Based Credit Risk Models & Benchmark Comparison
### Credit Risk & Loan Decision Intelligence Platform

---

## SECTION 1 — Step 2 Introduction

### 1. Context and Motivation
In credit risk analytics, evaluating borrower risk requires exploring both linear and non-linear modeling techniques. In **Phase 3 Step 1**, we established our initial benchmark using a linear **Logistic Regression** baseline. In this step (**Phase 3 Step 2**), we introduce two widely used tree-based ensemble algorithms: **Random Forest** and **XGBoost (Extreme Gradient Boosting)**, and perform a structured benchmark comparison against the baseline.

### 2. Why Explore Tree-Based Ensemble Models for Credit Risk?
- **Non-linear Decision Boundaries**: Real-world financial risk rarely follows pure linear relationships. For example, a high Debt-to-Income (DTI) ratio combined with an elevated Credit Utilization may multiply default risk exponentially rather than additively.
- **Automatic Feature Interaction**: Decision trees naturally partition feature space hierarchically (e.g., splitting on `credit_score` first, then evaluating `average_bank_balance` conditionally), capturing complex interactions without manual polynomial expansion.
- **Robustness to Monotonic Transforms & Outliers**: Tree splits depend on ordering rather than absolute feature distances.

### 3. Understanding the Candidate Models:
1. **Logistic Regression (Baseline Benchmark)**:
   - Models the log-odds of default as a linear combination of features.
   - Highly interpretable, regulator-friendly, convex optimization.
   - Serves as the control benchmark against which advanced models are evaluated.
2. **Random Forest Classifier (Bagging Ensemble)**:
   - Builds an ensemble of independent, de-correlated decision trees trained on bootstrap samples with random feature subsets.
   - Reduces variance and prevents overfitting compared to individual unpruned decision trees.
3. **XGBoost Classifier (Sequential Boosting Ensemble)**:
   - Builds decision trees sequentially, where each new tree is trained specifically to predict and correct the residual errors (pseudo-residuals) of the previous ensemble.
   - Incorporates second-order Taylor expansion gradients and $L_1/L_2$ tree regularization.

> **⚠️ Methodological Caution:**  
> We do **NOT** assume that tree-based models are automatically superior to Logistic Regression. The optimal algorithm depends on dataset scale, signal-to-noise ratio, feature geometry, and business constraints.

---
## SECTION 2 — Import Libraries & Reuse Phase 2 Data

We import required libraries and reproduce the exact certified Phase 2 transformation pipeline without modifying Phase 2 files or leaking test data:
- Stratified 80/20 train/test split with `random_state=42`.
- `StandardScaler` for 13 numerical columns (fitted strictly on `X_train`).
- `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` for `employment_type`.
- Resulting shapes verified: `X_train_processed` $(40, 16)$, `X_test_processed` $(10, 16)$.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load raw canonical dataset
data_path = '../data/raw/credit_risk_50.csv'
df = pd.read_csv(data_path)

target_col = 'default'
id_col = 'applicant_id'

y = df[target_col].copy()
ml_feature_cols = [col for col in df.columns if col not in [id_col, target_col]]
X = df[ml_feature_cols].copy()

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Numerical and Categorical feature definitions
numerical_cols = [
    'age', 'monthly_income', 'loan_amount', 'loan_term_months', 'existing_emi',
    'credit_score', 'credit_utilization_pct', 'late_payment_count', 'previous_defaults',
    'employment_years', 'average_bank_balance', 'monthly_expenses', 'debt_to_income_ratio'
]
categorical_cols = ['employment_type']

# Construct preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    verbose_feature_names_out=False
)

# Fit preprocessor strictly on X_train
preprocessor.fit(X_train)
feature_names = list(preprocessor.get_feature_names_out())

# Transform train and test matrices
X_train_processed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=feature_names,
    index=X_train.index
)
X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=feature_names,
    index=X_test.index
)

print(f"X_train_processed shape : {X_train_processed.shape} (Expected: 40 rows, 16 features)")
print(f"X_test_processed shape  : {X_test_processed.shape}  (Expected: 10 rows, 16 features)")
print(f"y_train shape           : {y_train.shape} (Expected: 40 labels)")
print(f"y_test shape            : {y_test.shape}  (Expected: 10 labels)\n")

assert X_train_processed.shape == (40, 16), "X_train_processed shape mismatch"
assert X_test_processed.shape == (10, 16), "X_test_processed shape mismatch"
assert y_train.shape == (40,), "y_train shape mismatch"
assert y_test.shape == (10,), "y_test shape mismatch"
print("Phase 2 Preprocessing Pipeline Replicated Successfully.")


X_train_processed shape : (40, 16) (Expected: 40 rows, 16 features)
X_test_processed shape  : (10, 16)  (Expected: 10 rows, 16 features)
y_train shape           : (40,) (Expected: 40 labels)
y_test shape            : (10,)  (Expected: 10 labels)

Phase 2 Preprocessing Pipeline Replicated Successfully.


---
## SECTION 3 — Load / Recreate Step 1 Baseline Model

We instantiate and fit the Step 1 baseline **Logistic Regression** model using `max_iter=1000, random_state=42` on `(X_train_processed, y_train)`.

In [2]:
# Baseline Logistic Regression model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_processed, y_train)

lr_pred = lr_model.predict(X_test_processed)
lr_proba = lr_model.predict_proba(X_test_processed)[:, 1]

print("Logistic Regression Baseline fitted successfully.")
print(f"Test Predictions (Discrete) : {lr_pred}")
print(f"Test Estimated Prob (PD)    : {np.round(lr_proba, 4)}")


Logistic Regression Baseline fitted successfully.
Test Predictions (Discrete) : [0 0 1 0 0 1 1 1 0 0]
Test Estimated Prob (PD)    : [9.330e-02 2.200e-03 9.746e-01 6.370e-02 5.000e-04 9.704e-01 9.791e-01
 9.210e-01 6.000e-04 1.755e-01]


---
## SECTION 4 — Random Forest Classifier

We train a `RandomForestClassifier` with a conservative, regularized baseline configuration suitable for small educational samples:
- `n_estimators=200`: Number of bagged trees.
- `max_depth=4`: Shallow depth limit to constrain tree complexity and prevent leaf memorization.
- `class_weight='balanced'`: Rebalances weights inversely proportional to class frequencies.
- `random_state=42`: Enforces exact reproducibility.

*(Note: Hyperparameter tuning is intentionally omitted in this step.)*

In [3]:
# Instantiate Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)

# Fit strictly on training data
rf_model.fit(X_train_processed, y_train)

# Generate test predictions and estimated probabilities
rf_pred = rf_model.predict(X_test_processed)
rf_proba = rf_model.predict_proba(X_test_processed)[:, 1]

print("Random Forest Classifier fitted successfully.")
print(f"RF Test Predictions (Discrete) : {rf_pred}")
print(f"RF Test Estimated Prob (PD)    : {np.round(rf_proba, 4)}")


Random Forest Classifier fitted successfully.
RF Test Predictions (Discrete) : [0 0 1 0 0 1 1 1 0 0]
RF Test Estimated Prob (PD)    : [0.1   0.    0.94  0.075 0.    1.    1.    0.97  0.    0.125]


---
## SECTION 5 — XGBoost Classifier

We train an `XGBClassifier` with a conservative, regularized configuration:
- `n_estimators=100`: Moderate ensemble size.
- `max_depth=3`: Shallow tree depth to limit high-order interaction complexity.
- `learning_rate=0.05`: Moderate shrinkage rate.
- `subsample=0.8`: Row subsampling to add stochastic regularization.
- `colsample_bytree=0.8`: Feature subsampling per tree split.
- `eval_metric='logloss'`: Binary cross-entropy optimization target.
- `random_state=42`: Reproducibility.

In [4]:
# Instantiate XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

# Fit strictly on training data
xgb_model.fit(X_train_processed, y_train)

# Generate test predictions and estimated probabilities
xgb_pred = xgb_model.predict(X_test_processed)
xgb_proba = xgb_model.predict_proba(X_test_processed)[:, 1]

print("XGBoost Classifier fitted successfully.")
print(f"XGB Test Predictions (Discrete) : {xgb_pred}")
print(f"XGB Test Estimated Prob (PD)    : {np.round(xgb_proba, 4)}")


XGBoost Classifier fitted successfully.
XGB Test Predictions (Discrete) : [0 0 1 0 0 1 1 1 0 0]
XGB Test Estimated Prob (PD)    : [0.0466 0.0426 0.9273 0.0466 0.0426 0.9278 0.9278 0.9278 0.0426 0.0466]


---
## SECTION 6 — Stratified 5-Fold Cross-Validation Comparison

### 🔬 Methodological Principle:
With only 40 training records and 10 test records, evaluating models solely on a single 10-sample test partition suffers from high sample variance. To provide a rigorous, statistically grounded evaluation, we perform **Stratified 5-Fold Cross-Validation strictly on the 40 training records** (`X_train_processed`, `y_train`).

- **Test Data Isolation**: `X_test` and `y_test` are strictly excluded from the cross-validation process.
- **Metrics Computed Across 5 Folds**: Accuracy, Precision, Recall, F1-Score, and ROC-AUC (using out-of-fold probability outputs).
- **Reported Statistics**: Mean score and standard deviation $(\mu \pm \sigma)$ across all 5 folds.

In [5]:
# Setup Stratified 5-Fold Cross-Validation on Training Data ONLY
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

candidate_models = {
    'Logistic Regression (Baseline)': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')
}

cv_summary = []

for model_name, model_obj in candidate_models.items():
    scores = cross_validate(
        model_obj,
        X_train_processed,
        y_train,
        cv=cv,
        scoring=scoring_metrics,
        return_train_score=False
    )
    
    cv_summary.append({
        'Model': model_name,
        'Accuracy (Mean +/- Std)': f"{scores['test_accuracy'].mean():.4f} +/- {scores['test_accuracy'].std():.4f}",
        'Precision (Mean +/- Std)': f"{scores['test_precision'].mean():.4f} +/- {scores['test_precision'].std():.4f}",
        'Recall (Mean +/- Std)': f"{scores['test_recall'].mean():.4f} +/- {scores['test_recall'].std():.4f}",
        'F1-Score (Mean +/- Std)': f"{scores['test_f1'].mean():.4f} +/- {scores['test_f1'].std():.4f}",
        'ROC-AUC (Mean +/- Std)': f"{scores['test_roc_auc'].mean():.4f} +/- {scores['test_roc_auc'].std():.4f}"
    })

cv_results_df = pd.DataFrame(cv_summary)
print("=== Stratified 5-Fold Cross-Validation Results (Training Set Only: 40 Rows) ===")
display(cv_results_df)


=== Stratified 5-Fold Cross-Validation Results (Training Set Only: 40 Rows) ===


,Model,Accuracy (Mean +/- Std),Precision (Mean +/- Std),Recall (Mean +/- Std),F1-Score (Mean +/- Std),ROC-AUC (Mean +/- Std)
0,Logistic Regression (Baseline),1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 +/- 0.0000
1,Random Forest,1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 +/- 0.0000,1.0000 +/- 0.0000
2,XGBoost,0.9500 +/- 0.1000,0.9333 +/- 0.1333,0.9333 +/- 0.1333,0.9333 +/- 0.1333,0.9867 +/- 0.0267


---
## SECTION 7 — Final Test Benchmark Comparison

Having completed cross-validation on the training set, we evaluate all three models on the untouched 10-record test set (`X_test_processed`, `y_test`). All metrics are computed dynamically from actual model predictions.

In [6]:
# Evaluate all 3 models on the 10-record test set
test_eval_records = []

models_fitted = {
    'Logistic Regression': (lr_pred, lr_proba),
    'Random Forest': (rf_pred, rf_proba),
    'XGBoost': (xgb_pred, xgb_proba)
}

for m_name, (preds, probas) in models_fitted.items():
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probas)
    
    test_eval_records.append({
        'Model': m_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4)
    })

test_benchmark_df = pd.DataFrame(test_eval_records)
print("=== Final Test Set Benchmark Comparison (10 Test Records) ===")
display(test_benchmark_df)


=== Final Test Set Benchmark Comparison (10 Test Records) ===


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,1.0,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0,1.0
2,XGBoost,1.0,1.0,1.0,1.0,1.0


---
## SECTION 8 — Confusion Matrices

We display formatted confusion matrices for all three models across the 10 test records.

In [7]:
for m_name, (preds, _) in models_fitted.items():
    cm = confusion_matrix(y_test, preds)
    cm_df = pd.DataFrame(
        cm,
        index=['Actual Non-Default (0)', 'Actual Default (1)'],
        columns=['Predicted Non-Default (0)', 'Predicted Default (1)']
    )
    print(f"\n=== Confusion Matrix: {m_name} ===")
    display(cm_df)



=== Confusion Matrix: Logistic Regression ===


,Predicted Non-Default (0),Predicted Default (1)
Actual Non-Default (0),6,0
Actual Default (1),0,4



=== Confusion Matrix: Random Forest ===


,Predicted Non-Default (0),Predicted Default (1)
Actual Non-Default (0),6,0
Actual Default (1),0,4



=== Confusion Matrix: XGBoost ===


,Predicted Non-Default (0),Predicted Default (1)
Actual Non-Default (0),6,0
Actual Default (1),0,4


---
## SECTION 9 — Model Interpretation & Feature Importance

### 1. Comparative Mechanics:
1. **Logistic Regression (Linear Link in Log-Odds)**:
   - Learns linear weights $\beta_i$. Each feature contributes independently to the log-odds of default without automatic interaction modeling.
   - Serves as the transparent baseline standard required by banking regulators.
2. **Random Forest (Bagged Ensembles)**:
   - Averages predictions across 200 de-correlated decision trees, capturing non-linear step functions and multi-variable interactions.
   - Computes feature importance via Mean Decrease in Impurity (Gini Impurity reduction).
3. **XGBoost (Gradient Boosted Trees)**:
   - Builds sequential trees that fit residual errors using second-order Taylor approximations and regularized objective functions.
   - Computes feature importance based on average gain across tree splits.

### 2. Feature Importance Display:
> **⚠️ CRITICAL CAUSALITY WARNING:**  
> Feature importances below represent **model-derived statistical associations** inside this specific dataset. They **DO NOT** imply direct physical or economic causality (e.g., having a high feature score does not automatically "cause" default; it indicates correlated predictive utility).

In [8]:
# Extract model-derived feature importances
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_names, name="RF Gini Importance")
xgb_importance = pd.Series(xgb_model.feature_importances_, index=feature_names, name="XGB Gain Importance")

importance_df = pd.DataFrame({
    'Random Forest Importance': rf_importance.round(4),
    'XGBoost Importance': xgb_importance.round(4)
}).sort_values(by='Random Forest Importance', ascending=False)

print("=== Model-Derived Feature Importances (Statistical Association Only) ===")
display(importance_df)


=== Model-Derived Feature Importances (Statistical Association Only) ===


,Random Forest Importance,XGBoost Importance
credit_utilization_pct,0.2276,0.2501
credit_score,0.1904,0.3282
late_payment_count,0.1664,0.2937
average_bank_balance,0.1167,0.0408
debt_to_income_ratio,0.1000,0.0000
previous_defaults,0.0673,0.0633
monthly_income,0.0369,0.0000
employment_years,0.0338,0.0237
age,0.0183,0.0000
monthly_expenses,0.0167,0.0000


---
## SECTION 10 — Model Comparison Conclusion

### 1. Summary of Experimental Findings:
- **5-Fold Cross-Validation (Training Set)**:
  - **Logistic Regression**: Mean CV Accuracy = 1.0000 $(\pm 0.0000)$, Mean ROC-AUC = 1.0000 $(\pm 0.0000)$.
  - **Random Forest**: Mean CV Accuracy = 1.0000 $(\pm 0.0000)$, Mean ROC-AUC = 1.0000 $(\pm 0.0000)$.
  - **XGBoost**: Mean CV Accuracy = 0.9500 $(\pm 0.1000)$, Mean ROC-AUC = 0.9867 $(\pm 0.0267)$.
- **Test Set Evaluation (10 Records)**:
  - All three models achieved 100% classification accuracy on the 10-record test set.

### 2. Analysis of Validation vs. Test Set Discrepancy:
In 5-fold CV on the training data, XGBoost showed a slight variation on one fold (8-sample validation split), producing a mean CV accuracy of 0.9500 compared to 1.0000 for Logistic Regression and Random Forest. This demonstrates the critical value of cross-validation: on extremely small datasets, a single test split can mask subtle variance across different data subsets.

### 3. Balanced Scientific Conclusion:
> Based on the current educational dataset and validation results, **Logistic Regression and Random Forest performed best on the 5-fold cross-validation metrics**, while all three models achieved comparable test performance.
>
> **Important Dataset Scope Disclaimer:**  
> This comparison is educational and architectural. The dataset contains **50 synthetic records** (40 train / 10 test), which is too small to establish production-level model performance or statistically definitive superiority between tree-based and linear models. In real-world enterprise underwriting on hundreds of thousands of records, tree models typically excel when complex nonlinear interactions dominate, whereas regularized linear models excel when linear bureau scorecards and monotonic credit attributes predominate.

---
## SECTION 11 — Automated Validation Suite

We execute 12 automated validation assertions certifying the complete pipeline.

In [9]:
print("Executing Phase 3 Step 2 Model Comparison Validation Gates...")

# 1. All three models fitted successfully
assert hasattr(lr_model, 'classes_'), "Gate 1 Failed: Logistic Regression is not fitted"
assert hasattr(rf_model, 'classes_'), "Gate 1 Failed: Random Forest is not fitted"
assert hasattr(xgb_model, 'classes_'), "Gate 1 Failed: XGBoost is not fitted"

# 2. Training matrix contains 16 features
assert X_train_processed.shape[1] == 16, "Gate 2 Failed: X_train_processed does not have 16 features"

# 3. Test matrix contains 16 features
assert X_test_processed.shape[1] == 16, "Gate 3 Failed: X_test_processed does not have 16 features"

# 4. Training records = 40
assert len(X_train_processed) == len(y_train) == 40, "Gate 4 Failed: Training count mismatch"

# 5. Test records = 10
assert len(X_test_processed) == len(y_test) == 10, "Gate 5 Failed: Test count mismatch"

# 6. Predictions contain exactly 10 records
assert len(lr_pred) == len(rf_pred) == len(xgb_pred) == 10, "Gate 6 Failed: Prediction counts != 10"

# 7. Probability arrays contain exactly 10 values
assert len(lr_proba) == len(rf_proba) == len(xgb_proba) == 10, "Gate 7 Failed: Probability counts != 10"

# 8. Probabilities are between 0 and 1
assert np.all((lr_proba >= 0.0) & (lr_proba <= 1.0)), "Gate 8 Failed: LR probabilities out of bounds"
assert np.all((rf_proba >= 0.0) & (rf_proba <= 1.0)), "Gate 8 Failed: RF probabilities out of bounds"
assert np.all((xgb_proba >= 0.0) & (xgb_proba <= 1.0)), "Gate 8 Failed: XGB probabilities out of bounds"

# 9. No NaN probabilities
assert not np.isnan(lr_proba).any() and not np.isnan(rf_proba).any() and not np.isnan(xgb_proba).any(), "Gate 9 Failed: NaN found in probabilities"

# 10. No infinite probabilities
assert not np.isinf(lr_proba).any() and not np.isinf(rf_proba).any() and not np.isinf(xgb_proba).any(), "Gate 10 Failed: Inf found in probabilities"

# 11. Cross-validation executed successfully
assert len(cv_summary) == 3, "Gate 11 Failed: Cross-validation summary incomplete"

# 12. Test evaluation executed successfully
assert len(test_eval_records) == 3, "Gate 12 Failed: Test evaluation records incomplete"

print("=" * 65)
print("PHASE 3 — STEP 2: MODEL COMPARISON VALIDATION PASSED")
print("=" * 65)


Executing Phase 3 Step 2 Model Comparison Validation Gates...
PHASE 3 — STEP 2: MODEL COMPARISON VALIDATION PASSED


---
## SECTION 12 — Completion Summary

### 📋 Phase 3 Step 2 Summary:

| Specification | Value | Context |
|---|---|---|
| **Models Compared** | Logistic Regression, Random Forest, XGBoost | 1 baseline + 2 tree ensembles |
| **Validation Methodology** | Stratified 5-Fold Cross-Validation | Performed strictly on 40 training records |
| **Test Evaluation** | Unseen 10-record Test Split | Evaluated on untouched test partition |
| **Input Features** | 16 features | 13 scaled numerical + 3 one-hot encoded |
| **Outputs Generated** | Hard class predictions & estimated PD probabilities | For all 3 models |
| **Validation Status** | ALL 12 GATES PASSED | Zero leakage, complete integrity |

---

```
============================================================
  PHASE 3 — STEP 1: COMPLETE
  PHASE 3 — STEP 2: COMPLETE
  PHASE 3 — STEP 3: NOT STARTED
============================================================
```